In [4]:
import os
from ensurepip import bootstrap
import warnings
from gc import set_debug
from statistics import mean

from jupyter_builder.jupyterlab_semver import satisfies
from sklearn.impute import SimpleImputer
from optuna.trial import FixedTrial
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from dotenv import load_dotenv, find_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score, f1_score, log_loss
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
import lightgbm as lgb
from catboost import CatBoostClassifier
import optuna
from sklearn.inspection import permutation_importance
import shap
from optuna.visualization import plot_param_importances, plot_optimization_history, plot_slice
import math


load_dotenv(find_dotenv(usecwd=True))
url = URL.create("postgresql+psycopg2",
    username=os.getenv("DB_USER"), password=os.getenv("DB_PASSWORD"),
    host="localhost", port=5433, database="adult")
engine = create_engine(url)

df = pd.read_sql("SELECT * FROM adults", engine)
df['workclass'] = df['workclass'].fillna('Private')
df['native_country'] = df['native_country'].fillna('United-States')
df = df.drop_duplicates()
top_6 = ['Prof-specialty',
    'Craft-repair',
    'Exec-managerial',
    'Adm-clerical',
    'Sales',
    'Other-service']
prods = df['occupation'].value_counts().loc[top_6]
prods = prods / prods.sum()
null_mask = df['occupation'].isna()
rng = np.random.default_rng(42)
df.loc[null_mask, 'occupation'] = rng.choice(top_6, size=null_mask.sum(), p=prods)

df.shape

(32537, 15)

In [ ]:
x = df.drop('income', axis=1)
y = df['income']